# 01. Kiểm tra dữ liệu trước khi chạy thí nghiệm

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.max_columns", None, "display.width", 200, "display.max_colwidth", 120)
from src.config import PATHS, load_config
import matplotlib.pyplot as plt
from shapely.geometry import box
from src.data.study_area import compute_bbox, districts_union, load_districts
from src.data.loaders import load_processed
from src.viz.expansion_maps import edges_projected


## 12 quận, bbox và các điểm kho/đích
Mọi điểm phải nằm trong bbox.

In [2]:
districts = load_districts()
bbox = compute_bbox()
print("bbox (west, south, east, north):", tuple(round(v, 5) for v in bbox))
loc = load_config("locations")
points = {k: v["latlon"] for k, v in {**loc["depots"], **loc["targets"]}.items()}
frame = box(*bbox)
from shapely.geometry import Point
inside = {k: frame.contains(Point(lon, lat)) for k, (lat, lon) in points.items()}
print("Mọi điểm nằm trong bbox:", all(inside.values()))
assert all(inside.values())
ax = districts.plot(color="#f2f1ec", edgecolor="#8a8984", figsize=(8, 7))
ax.plot(*frame.exterior.xy, color="#2a78d6", linewidth=1.5)
for k, (lat, lon) in points.items():
    ax.scatter(lon, lat, s=30, color="#d62728" if k.startswith("K") else "#0b0b0b", zorder=3)
    ax.annotate(k, (lon, lat), xytext=(4, 4), textcoords="offset points", fontsize=8)
ax.set_title("12 quận cũ, bbox và các điểm kho/đích")
plt.show()

bbox (west, south, east, north): (105.6768, 20.88377, 105.96698, 21.13214)
Mọi điểm nằm trong bbox: True


## Thống kê B1, B2 trước và sau LSCC
`node_keep_ratio` của B2 quyết định có bật `include_tertiary` hay không (ngưỡng 85%).

In [3]:
stats = pd.read_csv(PATHS.TABLES / "dataset_stats.csv").set_index("dataset")
display(stats.drop(columns=["highway_tags_before", "highway_tags_after"]).T)
ratio = stats.loc["B2", "node_keep_ratio"]
print(f"B2 giữ lại {ratio:.1%} số nút sau LSCC →", "giữ include_tertiary: false" if ratio >= 0.85 else "cần bật include_tertiary")

dataset,B1,B2
n_nodes_before,56459.000,4132.0000
n_edges_before,139362.000,7437.0000
total_km_before,13609.400,1901.4000
n_oneway_edges_before,11618.000,4713.0000
mean_out_degree_before,2.468,1.8000
n_nodes_after,56348.000,3890.0000
n_edges_after,139209.000,7156.0000
total_km_after,13573.900,1827.3000
n_oneway_edges_after,11503.000,4456.0000
mean_out_degree_after,2.471,1.8400


B2 giữ lại 94.1% số nút sau LSCC → giữ include_tertiary: false


In [4]:
graphs = {name: load_processed(name) for name in ["B1", "B2"]}
fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))
for ax, (name, G) in zip(axes, graphs.items()):
    edges_projected(G).plot(ax=ax, color="#52514e", linewidth=0.25 if name == "B1" else 0.5)
    ax.set_title(name); ax.set_axis_off()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\projects\\hanoi-delivery-search\\data\\processed\\B1.graphml'

## Kiểm tra cầu
Kỳ vọng: có cầu Chương Dương, Vĩnh Tuy, Thanh Trì, Nhật Tân trên cả B1 và B2; **không** có cầu Long Biên trên B1 (cấm ô tô).

In [ ]:
def edges_named(G, keyword):
    found = []
    for _, _, data in G.edges(data=True):
        name = data.get("name")
        names = name if isinstance(name, list) else [name]
        if any(keyword.lower() in str(n).lower() for n in names if n):
            found.append(data)
    return found

rows = []
for bridge, expected in [("Cầu Chương Dương", True), ("Cầu Vĩnh Tuy", True), ("Cầu Thanh Trì", True), ("Cầu Nhật Tân", True), ("Cầu Long Biên", False)]:
    row = {"cầu": bridge, "kỳ vọng có": expected}
    for name, G in graphs.items():
        hits = edges_named(G, bridge)
        row[f"số cạnh {name}"] = len(hits)
    rows.append(row)
bridges = pd.DataFrame(rows)
bridges

,cầu,kỳ vọng có,số cạnh B1,số cạnh B2
0,Cầu Chương Dương,True,15,15
1,Cầu Vĩnh Tuy,True,7,5
2,Cầu Thanh Trì,True,18,6
3,Cầu Nhật Tân,True,12,11
4,Cầu Long Biên,False,0,0


## Tag `highway` của Vành đai 3 trên cao
Bộ lọc B2 phải giữ được tuyến này.

In [ ]:
from collections import Counter
for name, G in graphs.items():
    tags = Counter()
    km = 0.0
    for data in edges_named(G, "Vành đai 3"):
        tag = data.get("highway")
        tags.update(tag if isinstance(tag, list) else [tag])
        km += data["length"] / 1000
    print(f"{name}: Vành đai 3 dài {km:.1f} km (tổng các chiều), tag {dict(tags)}")

B1: Vành đai 3 dài 67.5 km (tổng các chiều), tag {'motorway': 33, 'motorway_link': 7, 'residential': 4, 'primary': 33, 'secondary': 49}
B2: Vành đai 3 dài 63.3 km (tổng các chiều), tag {'motorway': 33, 'motorway_link': 7, 'primary': 18, 'secondary': 22}
